<a href="https://colab.research.google.com/github/neuromatch/course-kit-guide/blob/notebooks-branch/notebooks/guide/examples/directives/video-and-slides.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/neuromatch/course-kit-guide/notebooks-branch/notebooks/guide/examples/directives/video-and-slides.ipynb" target="_blank"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

# Example: Video and slides

For: author (example source)

This page is the source behind the "You write" blocks on [Videos and slides](../../author/videos-and-slides.md). It has a `kernelspec` because only pages with one can use `:path:` for a PDF in the repository.

% region-start: video

In [ ]:
# @title Video 1: Linear Dynamical Systems
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', '87z6OR7-DBI'), ('Bilibili', 'BV1up4y1S7wj')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

% region-end: video

% region-start: slides

In [ ]:
# @title Tutorial slides
import json, sys, uuid
from IPython.display import HTML, IFrame, display
from ipywidgets import widgets

# Colab shows cell output in a sandboxed frame, where a browser can't draw a bare
# PDF. There the PDF is drawn with PDF.js; elsewhere it is embedded as is.
IN_COLAB = "google.colab" in sys.modules

PDF_VIEWER = r"""<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/pdfjs-dist@4.10.38/web/pdf_viewer.css">
<div id="__ID__" style="width:854px;max-width:100%;height:480px;display:flex;flex-direction:column;border:1px solid #ccc;box-sizing:border-box;background:#eee;font:14px sans-serif">
  <div data-bar style="display:flex;gap:6px;align-items:center;padding:4px 6px;background:#f8f8f8;border-bottom:1px solid #ccc">
    <button data-act="out" title="Zoom out">-</button>
    <button data-act="in" title="Zoom in">+</button>
    <button data-act="fit" title="Fit the whole slide">Fit</button>
    <button data-act="width" title="Fit the width">Width</button>
    <span style="flex:1"></span>
    <button data-act="prev" title="Previous slide">&lt;</button>
    <span data-page style="min-width:4em;text-align:center">Loading...</span>
    <button data-act="next" title="Next slide">&gt;</button>
  </div>
  <div style="position:relative;flex:1;min-height:0">
    <div data-container style="position:absolute;inset:0;overflow:auto"><div class="pdfViewer"></div></div>
  </div>
</div>
<script>
(async () => {
  const root = document.getElementById("__ID__");
  const url = __URL__;
  const lib = "https://cdn.jsdelivr.net/npm/pdfjs-dist@4.10.38/";
  try {
    const pdfjs = await import(lib + "build/pdf.min.mjs");
    window.pdfjsLib = pdfjs;
    const viewerLib = await import(lib + "web/pdf_viewer.mjs");
    const worker = await (await fetch(lib + "build/pdf.worker.min.mjs")).text();
    pdfjs.GlobalWorkerOptions.workerSrc = URL.createObjectURL(new Blob([worker], {type: "text/javascript"}));

    const container = root.querySelector("[data-container]");
    const label = root.querySelector("[data-page]");
    const eventBus = new viewerLib.EventBus();
    const linkService = new viewerLib.PDFLinkService({eventBus});
    const viewer = new viewerLib.PDFViewer({container, viewer: container.firstElementChild, eventBus, linkService});
    linkService.setViewer(viewer);

    let mode = "page-fit";
    const show = (value) => { mode = value; viewer.currentScaleValue = value; };
    const zoom = (factor) => { mode = null; viewer.currentScale = viewer.currentScale * factor; };
    eventBus.on("pagesinit", () => {
      show("page-fit");
      label.textContent = "1 / " + viewer.pagesCount;
    });
    eventBus.on("pagechanging", (event) => {
      label.textContent = event.pageNumber + " / " + viewer.pagesCount;
    });
    root.querySelector("[data-bar]").addEventListener("click", (event) => {
      const button = event.target.closest("button");
      const actions = {
        out: () => zoom(1 / 1.25),
        in: () => zoom(1.25),
        fit: () => show("page-fit"),
        width: () => show("page-width"),
        prev: () => viewer.previousPage(),
        next: () => viewer.nextPage(),
      };
      if (button && actions[button.dataset.act]) actions[button.dataset.act]();
    });
    new ResizeObserver(() => { if (mode) viewer.currentScaleValue = mode; }).observe(root);

    const pdf = await pdfjs.getDocument({
      url,
      cMapUrl: lib + "cmaps/",
      cMapPacked: true,
      standardFontDataUrl: lib + "standard_fonts/",
    }).promise;
    viewer.setDocument(pdf);
    linkService.setDocument(pdf, null);
  } catch (error) {
    console.error(error);
    root.textContent = "Could not show the slides here. ";
    const link = document.createElement("a");
    link.href = url;
    link.target = "_blank";
    link.textContent = "Open the PDF in a new tab";
    root.appendChild(link);
  }
})();
</script>"""


def show_pdf(url):
  box = "nma-slides-" + uuid.uuid4().hex[:12]
  display(HTML(PDF_VIEWER.replace("__ID__", box).replace("__URL__", json.dumps(url))))


# (label, embed url, download url, the embed url is a bare PDF)
slides = [('OSF', 'https://mfr.ca-1.osf.io/render?url=https://osf.io/snv4m/?direct%26mode=render%26action=download%26mode=render', 'https://osf.io/download/snv4m/', False), ('PDF', 'https://cdn.jsdelivr.net/gh/neuromatch/course-kit-guide@notebooks-branch/notebooks/guide/examples/directives/slides/example_slides.pdf', 'https://cdn.jsdelivr.net/gh/neuromatch/course-kit-guide@notebooks-branch/notebooks/guide/examples/directives/slides/example_slides.pdf', True)]

tab_contents = []
for label, src, download, is_pdf in slides:
  out = widgets.Output()
  with out:
    print(f"If you want to download the slides: {download}")
    if is_pdf and IN_COLAB:
      show_pdf(src)
    else:
      display(IFrame(src=src, width=854, height=480))
  tab_contents.append(out)
if len(tab_contents) == 1:
  display(tab_contents[0])
else:
  tabs = widgets.Tab()
  tabs.children = tab_contents
  for i, (label, _, _, _) in enumerate(slides):
    tabs.set_title(i, label)
  display(tabs)

% region-end: slides